# Imports

In [2]:
import json
from typing import List

# Unstructured for document parsing
from unstructured.partition.pdf import partition_pdf
from unstructured.chunking.title import chunk_by_title

# LangChain components
from langchain_core.documents import Document
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_chroma import Chroma
from langchain_core.messages import HumanMessage
from dotenv import load_dotenv

load_dotenv()


c:\Users\Pratikesh\Documents\VS_CODE\LLM_RAG_practices\RAG_Langchain\myenv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


True

# Partition Document

In [3]:
def partition_document(file_path: str):
    print(f"Document:{file_path}")

    elements = partition_pdf(
        filename=file_path, #file Path
        strategy="hi_res", #most accurate (but slower) processing method of extraction
        infer_table_structure=True, #for tables , not stored as jumbled text
        extract_image_block_types=['Image'], # for image
        extract_image_block_to_payload=True,  # store image  as base64
        languages=["English"]   
        )

    print(f"Extracted {len(elements)} elements")
    return elements

file_path = "data/New.pdf"
elements = partition_document(file_path)


Document:data/New.pdf


Cannot set non-stroke color because expected 4 components but got [0]
Cannot set non-stroke color because expected 4 components but got [0]
Cannot set non-stroke color because expected 4 components but got [0]
Cannot set non-stroke color because expected 4 components but got [0]
Cannot set non-stroke color because expected 4 components but got [0]
Cannot set non-stroke color because expected 4 components but got [0]
Loading weights: 100%|██████████| 367/367 [00:00<00:00, 4316.93it/s]


Extracted 223 elements


In [4]:
elements[3].to_dict()


{'type': 'NarrativeText',
 'element_id': '8d742515785136044f6fa5486a39b020',
 'text': 'An enormous variety of organisms exist, including some which can survive and even develop in the body of people or animals. If the organism can cause infection, it is an infectious agent. In this manual infectious agents which cause infection and illness are called pathogens. Diseases caused by pathogens, or the toxins they produce, are communicable or infectious diseases (45). In this manual these will be called disease and infection.',
 'metadata': {'detection_class_prob': 0.9546390175819397,
  'is_extracted': 'true',
  'coordinates': {'points': ((np.float64(274.8428955078125),
     np.float64(1061.49609375)),
    (np.float64(274.8428955078125), np.float64(1463.7243055555552)),
    (np.float64(2157.815673828125), np.float64(1463.7243055555552)),
    (np.float64(2157.815673828125), np.float64(1061.49609375))),
   'system': 'PixelSpace',
   'layout_width': 2426,
   'layout_height': 3447},
  'last_mod

In [5]:
## Gather All IMGs

imgs = [element for element in elements if element.category == 'Image']
print(f"Found {len(imgs)} images")


Found 6 images


In [6]:
imgs[0].to_dict()
# Use https://codebeautify.org/base64-to-image-converter to view the base64 text

{'type': 'Image',
 'element_id': 'baeb32455985db605887c4eac4e430e5',
 'text': 'The environment  Entry of the pathogen  Transmission  The pathogen leaves  the host  The susceptible person  or animal  The host  The pathogen ',
 'metadata': {'detection_class_prob': 0.8926597237586975,
  'coordinates': {'points': ((np.float64(289.0517883300781),
     np.float64(1122.2371826171875)),
    (np.float64(289.0517883300781), np.float64(2948.26904296875)),
    (np.float64(2163.435791015625), np.float64(2948.26904296875)),
    (np.float64(2163.435791015625), np.float64(1122.2371826171875))),
   'system': 'PixelSpace',
   'layout_width': 2426,
   'layout_height': 3447},
  'last_modified': '2026-04-29T12:00:14',
  'filetype': 'application/pdf',
  'languages': ['eng'],
  'page_number': 2,
  'image_base64': '...',
  'image_mime_type': 'image/jpeg',
  'file_directory': 'data',
  'filename': 'New.pdf'}}

In [7]:
## Gather All Tables
tables = [element for element in elements if element.category == 'Table']
print(f"Found {len(tables)} tables")

tables[0].to_dict()

# Use https://jsfiddle.net/ to view the table html 

Found 3 tables


{'type': 'Table',
 'element_id': '61d75760ceddb76ba2694296eccacf9d',
 'text': 'Pathogen Description Latency Persistence Virus Particles invade living The pathogens Viruses can cells. The pathogen are non-latent. survive for needs structures in months in these cells to tropical reproduce. (45) temperatures. (28) Rickettsiae Organisms resemble n/a n/a bacteria. (45). However, similar to viruses, the pathogen needs to develop inside the cells of the host.(2) Bacteria Bacteria are single cell The pathogens Persists up to organisms. They are are non-latent. several weeks. considered more (16,73). Can primitive than animal multiply outside or plant cells. (45) the host. (3) Fungi A group of organisms n/a n/a which include yeast, moulds, and mushrooms. (45) Protozoa Protozoa area single The pathogens Forms a cell organisms. (45) are non-latent. resistant cyst which can survive for months. (3,44) Helminths Helminths are worms The pathogen is The pathogen is (worms) (roundworms, flukes or laten

# Create Chunks By title

In [8]:
def create_chunks(elements):
    print(f"Creating smart chunks..")

    chunks = chunk_by_title(
        elements,
        max_characters=3000,
        new_after_n_chars=2400,
        combine_text_under_n_chars=500 # Merge tiny chunks under 500 chars with neighbors
    )

    print(f"Created {len(chunks)} chunks")
    return chunks

#Chunks
chunks = create_chunks(elements)

Creating smart chunks..
Created 31 chunks


In [9]:
# View a single chunk
chunks[2].to_dict()

{'type': 'CompositeElement',
 'element_id': '05808ace-10df-46f2-846a-ccc1a4a2dc10',
 'text': '2.2 The pathogen\n\nThe pathogen is the organism that causes the infection.* Specific pathogens cause specific infections. Cholera is caused by the bacterium Vibrio cholerae, for example, and Leishmaniasis is caused by different species (spp.) of the protozoa Leishmania.\n\nSpecific infections also have specific transmission cycles. To be able to react appropriately to health problems in a population, the specific infection causing the problems must be known. Identification of the infection will usually be done by medical personnel.\n\nDifferent categories of pathogens can infect humans. The pathogens causing the diseases covered in this manual include viruses, bacteria, rickettsiae, fungi, proto- zoa, and helminths (worms). All pathogens go through a lifecycle, which takes the organism from reproducing adult to reproducing adult. This cycle includes phases of growth, consolidation, change of 

# Seprate Content type (based on Text,Img,table)

In [13]:
# Function to seprate the content  based on text,img,tables

def sep_content_types(chunk):
    content_data = {
        'text':chunk.text,
        'tables':[],
        'Images': [],
        'types' : ['text']
    }

    # checks for tables and images
    if hasattr(chunk,'metadata') and hasattr(chunk.metadata, 'orig_elements'):
        for element in chunk.metadata.orig_elements:
            element_type = type(element).__name__

            #Handle Table
            if element_type == 'Table':
                content_data['tables'].append('table')
                table_html = getattr(element.metadata,'text_as_html',element.text)
                content_data['tables'].append(table_html)

            #Handle Images
            elif element_type == 'Image':
                if hasattr(element,'metadata') and hasattr(element.metadata,'image_bade64'):
                    content_data['types'].append('Image')
                    content_data['Images'].append(element.metadata.image_base64)
            
    content_data['types'] = list(set(content_data['types']))
    return content_data



In [28]:
for i in range(len(chunks)):
    print(sep_content_types(chunks[i]))

{'text': 'DISEASE AND DISEASE TRANSMISSION\n\nChapter 2\n\nDisease and disease transmission\n\nAn enormous variety of organisms exist, including some which can survive and even develop in the body of people or animals. If the organism can cause infection, it is an infectious agent. In this manual infectious agents which cause infection and illness are called pathogens. Diseases caused by pathogens, or the toxins they produce, are communicable or infectious diseases (45). In this manual these will be called disease and infection.\n\nThis chapter presents the transmission cycle of disease with its different elements, and categorises the different infections related to WES.', 'tables': [], 'Images': [], 'types': ['text']}
{'text': '2.1 Introduction to the transmission cycle of disease\n\nTo be able to persist or live on, pathogens must be able to leave an infected host, survive transmission in the environment, enter a susceptible person or animal, and develop and/or multiply in the newly 

In [29]:
# Writing the Ai enchancd summary for Tables And Images
def Ai_summary(text:str,tables:List[str],images:list[str]) -> str:
    
    
    try:
        #initialize LLM
        llm=ChatOpenAI(model = 'gpt-4o',temperature=0)

        #build the text prompt
        prompt_text = f"""You are creating a searchable description for document content retrieval.

        CONTENT TO ANALYZE:
        TEXT CONTENT:
        {text}

        """

        #Add tables if Present
        if tables:
            prompt_text +="TABLES:\n"
            for i,table in  enumerate(tables):
                prompt_text +=f"tablle {i+1}:\n{table}\n\n"

                prompt_text += """
                YOUR TASK:
                Generate a comprehensive, searchable description that covers:

                1. Key facts, numbers, and data points from text and tables
                2. Main topics and concepts discussed  
                3. Questions this content could answer
                4. Visual content analysis (charts, diagrams, patterns in images)
                5. Alternative search terms users might use

                Make it detailed and searchable - prioritize findability over brevity.

                SEARCHABLE DESCRIPTION:"""
        
        # Build message content starting with text
        message_content = [{"type": "text", "text": prompt_text}]

        # Add images to the message
        for image_base64 in images:
            message_content.append({
                "type": "image_url",
                "image_url": {"url": f"data:image/jpeg;base64,{image_base64}"}
            })
        
        # Send to AI and get response
        message = HumanMessage(content=message_content)
        response = llm.invoke([message])
        
        return response.content



    except Exception as e:
        print(f"     ❌ AI summary failed: {e}")
        # Fallback to simple summary
        summary = f"{text[:300]}..."
        if tables:
            summary += f" [Contains {len(tables)} table(s)]"
        if images:
            summary += f" [Contains {len(images)} image(s)]"
        return summary


In [33]:
content_data = sep_content_types(chunks[0])

enhanced_content = Ai_summary(
    content_data['text'],
    content_data['tables'],
    content_data['Images']
)
                

     ❌ AI summary failed: Error code: 401 - {'error': {'message': 'Incorrect API key provided: s***. You can find your API key at https://platform.openai.com/account/api-keys.', 'type': 'invalid_request_error', 'param': None, 'code': 'invalid_api_key'}}
